In [1]:
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_networkx
import networkx as nx

In [2]:
PATH   = "../data/"

pubmed = Planetoid(PATH, "PubMed")

pubmed_nx = to_networkx(pubmed[0], node_attrs=['x', 'y'], to_undirected=True)

del pubmed

In [3]:
from sklearn.model_selection import train_test_split
import pandas as pd

In [4]:
df = pd.DataFrame(pubmed_nx.nodes.values())
x = pd.DataFrame(df['x'].to_list(), index=df['x'].index)
x.columns = x.columns.map(lambda x: f"w_{x}" if isinstance(x, int) else x)
y = df["y"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [5]:
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, f1_score, precision_score

In [6]:
def report(model, x_test, y_test):
    y_pred = model.predict(x_test)

    accuracy = accuracy_score(y_test, y_pred)
    
    precision = precision_score(
        y_test, y_pred, average="weighted"
    )
    
    recall = recall_score(
        y_test, y_pred, average="weighted"
    )
    
    f1 = f1_score(
        y_test, y_pred, average="macro"
    ) 
    
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"Accuracy : {accuracy:.4f} ")
    print(f"Precision : {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 : {f1:.4f}")
    
    print("-" * 40)
    
    print("Confusion Matrix:")
    print(f"TN: {cm[0][0]:<5} | FP: {cm[0][1]}")
    print(f"FN: {cm[1][0]:<5} | TP: {cm[1][1]}")

In [7]:
from sklearn.ensemble import RandomForestClassifier

In [8]:
rfc = RandomForestClassifier()
rfc.fit(x_train, y_train)
report(rfc, x_test, y_test)

Accuracy : 0.8887 
Precision : 0.8890
Recall   : 0.8887
F1 : 0.8881
----------------------------------------
Confusion Matrix:
TN: 718   | FP: 24
FN: 39    | TP: 1409


In [9]:
from sklearn.svm import SVC

In [10]:
svc = SVC()
svc.fit(x_train, y_train)
report(svc, x_test, y_test)

Accuracy : 0.8798 
Precision : 0.8798
Recall   : 0.8798
F1 : 0.8794
----------------------------------------
Confusion Matrix:
TN: 717   | FP: 27
FN: 50    | TP: 1367


In [11]:
nx_graph = pubmed_nx
tfidf_matrix = x.to_numpy()

In [12]:
import networkx as nx
import numpy as np
from scipy.sparse.csgraph import dijkstra
from sklearn.metrics.pairwise import cosine_distances

adj_matrix = nx.to_scipy_sparse_array(nx_graph)
D_G = dijkstra(csgraph=adj_matrix, directed=False, unweighted=True)

max_finite_dist = np.max(D_G[np.isfinite(D_G)])
D_G[np.isinf(D_G)] = max_finite_dist * 2

D_G /= np.max(D_G)

D_T = cosine_distances(tfidf_matrix)

alpha = 0.4
D_hybrid = alpha * D_G + (1.0 - alpha) * D_T

In [13]:
from sklearn.decomposition import TruncatedSVD

pagerank_dict = nx.pagerank(nx_graph)
lens_structural = np.array([pagerank_dict[i] for i in range(len(nx_graph))]).reshape(-1, 1)

svd = TruncatedSVD(n_components=1, random_state=42)
lens_semantic = svd.fit_transform(tfidf_matrix)

global_lens = np.column_stack([lens_structural, lens_semantic])

In [14]:
from sklearn.base import BaseEstimator, TransformerMixin, ClusterMixin
from sklearn.cluster import DBSCAN

class IndexedLensTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lens_array):
        self.lens_array = lens_array
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        indices = X.flatten().astype(int)
        return self.lens_array[indices]

class IndexedPrecomputedDBSCAN(BaseEstimator, ClusterMixin):
    def __init__(self, distance_matrix, eps=0.15, min_samples=3):
        self.distance_matrix = distance_matrix
        self.eps = eps
        self.min_samples = min_samples
    
    def fit(self, X, y=None):
        indices = X.flatten().astype(int)
        if len(indices) == 0:
            self.labels_ = np.array([])
            return self
            
        local_dist = self.distance_matrix[indices][:, indices]
        
        db = DBSCAN(metric='precomputed', eps=self.eps, min_samples=self.min_samples)
        db.fit(local_dist)
        self.labels_ = db.labels_
        return self

In [15]:
from gtda.mapper import make_mapper_pipeline, CubicalCover, plot_static_mapper_graph

X_indices = np.arange(len(nx_graph)).reshape(-1, 1)

cover = CubicalCover(n_intervals=20, overlap_frac=0.2)

filter_transformer = IndexedLensTransformer(global_lens)
custom_clusterer = IndexedPrecomputedDBSCAN(D_hybrid, eps=0.20, min_samples=8)

pipeline = make_mapper_pipeline(
    filter_func=filter_transformer,
    cover=cover,
    clusterer=custom_clusterer,
    contract_nodes=True
)

mapper_graph = pipeline.fit_transform(X_indices)

In [16]:
plot_static_mapper_graph(
    pipeline, 
    X_indices, 
    color_data=global_lens[:, 1]
)

FigureWidget({
    'data': [{'hoverinfo': 'none',
              'line': {'color': '#888', 'width': 1},
       …

In [17]:
import numpy as np
import scipy.sparse as sp

node_elements = mapper_graph.vs["node_elements"]

num_papers = len(nx_graph)
num_mapper_nodes = len(node_elements)

rows = []
cols = []

for node_idx, paper_indices in enumerate(node_elements):
    for p_idx in paper_indices:
        rows.append(p_idx)
        cols.append(node_idx)

X_topo = sp.csr_matrix(
    (np.ones(len(rows)), (rows, cols)), 
    shape=(num_papers, num_mapper_nodes), 
    dtype=np.float32
)

In [18]:
from sklearn.model_selection import train_test_split

X_combined = sp.hstack([tfidf_matrix, X_topo], format='csr')

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
)

rfc2 = RandomForestClassifier()
rfc2.fit(X_train, y_train)

report(rfc2, X_test, y_test)

Accuracy : 0.8879 
Precision : 0.8886
Recall   : 0.8879
F1 : 0.8874
----------------------------------------
Confusion Matrix:
TN: 722   | FP: 25
FN: 45    | TP: 1412


In [19]:
sorted_indices = np.argsort(rfc2.feature_importances_)[::-1]
sorted_indices[:30]

array([  0,  16, 401,  70, 184, 139, 449,  89, 444, 379, 239, 346, 235,
         8,  69, 123, 500, 477, 251,  21, 286, 359, 329,  60, 196,  44,
       247,  38, 386,  15])

In [20]:
rfc2.feature_importances_[sorted_indices[:30]]

array([0.05881384, 0.04719746, 0.03476555, 0.03215674, 0.02818566,
       0.01825917, 0.01819765, 0.01702079, 0.0163572 , 0.01524562,
       0.0145596 , 0.01418674, 0.0140557 , 0.01163388, 0.01037599,
       0.01024972, 0.01020437, 0.00760928, 0.00688004, 0.00687499,
       0.00686722, 0.00658849, 0.00647727, 0.0061164 , 0.0057789 ,
       0.00571817, 0.00568372, 0.00566165, 0.00534021, 0.00534008])

In [21]:
sorted_indices = np.argsort(rfc.feature_importances_)[::-1]
sorted_indices[:30]

array([  0,  16,  70, 401, 184, 139, 449, 444, 239,  89, 379, 235,   8,
       346,  69, 477, 123,  21, 286, 196, 359, 329,  44,  15, 247, 251,
        38, 386, 415, 448])

In [22]:
rfc.feature_importances_[sorted_indices[:30]]

array([0.07587041, 0.04638366, 0.03546738, 0.03535405, 0.03096046,
       0.0206507 , 0.01984801, 0.01937596, 0.01914458, 0.01841536,
       0.01836774, 0.01441595, 0.01284774, 0.01174355, 0.0103616 ,
       0.00903437, 0.00805912, 0.00758617, 0.0074023 , 0.00705135,
       0.00685699, 0.00684774, 0.00626065, 0.00610985, 0.00607539,
       0.00592654, 0.00558921, 0.00464646, 0.00458789, 0.0043657 ])